# Qwen3-14B — reasoning ON

One model, one configuration — built to run in its own Colab session alongside the others. Results land in `results/` and download as a zip at the end.

**This is the slow one, and the one that failed last time.** At a 512-token budget it produced no parseable answer on 91 of 210 prompts, because generation was cut off mid-reasoning. The budget is now 2048, which should fix it but makes the run considerably longer — budget 3–6 hours.

**Watch the parse rate in the quality gate at the end.** If it still fails, the numbers are not usable: an unanswered prompt is missing data, and the arms stop being scored on the same scenarios. Raise `MAX_GEN_TOKENS` and re-run rather than reading them.

Reasoning runs are behavioural-only — with a reasoning block in the way, the answer is no longer at the final prompt position for patching to reach.

## 1. Repository

In [ ]:
import os, sys, json, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    if subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                      capture_output=True, text=True).stdout.strip():
        !git -C $REPO_DIR stash -u
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
!git -C $REPO_DIR log --oneline -1

## 2. Install

In [ ]:
%pip install -q -e '/content/behaviour-microscope'
print("installed")

## 3. GPU

In [ ]:
import torch

NEEDED_GB = 30
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
p = torch.cuda.get_device_properties(0)
total = p.total_memory / 1e9
print(f"{p.name}  |  {total:.0f} GB  |  compute {p.major}.{p.minor}")
assert total > NEEDED_GB + 4, f"Needs ~{NEEDED_GB} GB of weights plus headroom."

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda"
print("dtype:", DTYPE, " device:", DEVICE)

## 4. Configure

In [ ]:
from microscope.experiment import RunConfig, run_all
from microscope.scenarios import ARMS

MODEL_ID = "Qwen/Qwen3-14B"
ENABLE_THINKING = True
MAX_GEN_TOKENS = 2048   # raise if the parse rate still fails

cfg = RunConfig(
    model_id=MODEL_ID, provider="local", backend="eager", dtype=DTYPE,
    extra_load_kwargs={"device": DEVICE},
    enable_thinking=ENABLE_THINKING,
    n_candidate_layers=4,
    max_gen_tokens=MAX_GEN_TOKENS,
    # arms=("floor", "junior_said", "partner_said", "partner_confirmed", "court"),
    #   ^ five arms instead of seven cuts a slow run by ~30%
)

for arm in ARMS:
    print(f"  {arm.name:20s} {arm.cue or '(no assertion)'}")
print()
print("model:", MODEL_ID, "| reasoning:", ENABLE_THINKING)


## 5. Run

In [ ]:
import concurrent.futures

# interp-engine's sync facade refuses to run inside an already-running event loop,
# and Colab's kernel keeps one running for every cell. A worker thread has none.
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    run_dir = pool.submit(run_all, cfg).result()
run_dir

## 6. Read the quality gate first

A `FAIL` means the numbers below it are not usable. `parse_rate` is the one to watch on a reasoning run: an unparseable response is missing data, excluded from every rate, so the arms stop being scored on the same scenarios.

In [ ]:
report = json.loads((run_dir / "quality_report.json").read_text())
print("Overall:", report["overall"].upper(), "\n")
for c in report["checks"]:
    print(f"[{c['status'].upper():4s}] {c['name']}: {c['detail']}")

In [ ]:
s = json.loads((run_dir / "summary.json").read_text())["behavioural"]
print(f"{'arm':22s} {'accepts false':>14s} {'accuracy':>9s} {'n scored':>9s}")
for arm in ["floor","junior_said","junior_confirmed","partner_said",
            "partner_confirmed","court","adverse"]:
    if arm not in s["fpar_by_arm"]: continue
    print(f"  {arm:20s} {s['fpar_by_arm'][arm]:13.0%} "
          f"{s['accuracy_by_arm'].get(arm, float('nan')):8.0%} "
          f"{s['n_scored_by_arm'][arm]:8d}/30")
print(f"\nparse failures: {s['parse_failures']}/{s['n_measurements']}")

## 7. Download

In [ ]:
import shutil
archive = shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"Download from the file browser instead ({exc})")